# Chapter 23 Exercises: Natural Language Processing

This notebook contains hands-on exercises for Chapter 23, "Natural Language Processing".

You will practice:

- n-gram language modeling and smoothing
- bag-of-words classification
- part-of-speech tagging with Viterbi decoding
- probabilistic context-free grammar parsing
- augmented grammars with agreement features
- semantic interpretation
- quantifier-scope ambiguity
- information extraction and simple question answering

Work through the TODO sections. Run the test cells after each exercise.

## 0. Setup

Run this cell first. The notebook uses only the Python standard library.

In [1]:
import math
import random
import re
from collections import Counter, defaultdict

# Cố định seed để các kết quả ngẫu nhiên, nếu có, dễ lặp lại khi chấm bài hoặc demo.
random.seed(11)


def show_rows(rows, headers=None):
    # Hàm phụ trợ này in dữ liệu dạng bảng đơn giản, không cần cài thêm thư viện ngoài.
    if headers:
        print(" | ".join(headers))
        print("-" * (3 + sum(len(h) for h in headers)))
    for row in rows:
        print(" | ".join(str(x) for x in row))


corpus = [
    "the wumpus is near .",
    "the pit is near .",
    "the gold is glittering .",
    "the agent grabs the gold .",
    "the agent smells the wumpus .",
    "the breeze means the pit is near .",
    "the stench means the wumpus is near .",
]

## Exercise 1: Tokenization and N-gram Language Models

Implement a lowercase tokenizer, then train a bigram or trigram model from the toy corpus.

Requirements:

- `tokenize(text)` should return words and sentence punctuation such as `"."`
- `train_ngram(sentences, n)` should count contexts and n-grams
- `next_word_probability(...)` should support add-k smoothing
- `sentence_log_probability(...)` should return `-inf` when an unsmoothed probability is zero

In [2]:
def tokenize(text):
    # TODO: Return lowercase tokens. Keep ".", "!", and "?" as separate tokens.
    # Gợi ý: dùng re.findall(...) để tách word token và dấu câu, rồi lower() để chuẩn hóa chữ hoa/thường.
    tokenized = re.findall(r"[a-zA-Z]+|[.!?]", text)
    return [t.lower() for t in tokenized]


def train_ngram(sentences, n):
    # TODO: Return a dictionary with keys: n, context_counts, ngram_counts.
    # Hint: pad each sentence with n-1 start symbols.
    # Gợi ý: Counter rất phù hợp để đếm số lần xuất hiện của context và n-gram.
    # Với mỗi vị trí i, context là n-1 token đứng trước token hiện tại.
    context_counts = Counter()
    ngram_counts = Counter()

    for sent in sentences:
        tokens = ["<s>"] * (n - 1) + tokenize(sent) + ["</s>"]
        for i in range(n - 1, len(tokens)):
            context = tuple(tokens[i - n + 1:i])
            word = tokens[i]

            context_counts[context] += 1
            ngram_counts[context + (word,)] += 1

    return {
        "n": n,
        "context_counts": context_counts,
        "ngram_counts": ngram_counts,
    }        


def next_word_probability(model, context, word, vocabulary, k=0.0):
    # TODO: Estimate P(word | context) with add-k smoothing.
    # Công thức add-k smoothing: (count(context, word) + k) / (count(context) + k * |V|).
    # Cần cắt context về đúng độ dài n-1 vì người dùng có thể truyền vào context dài hơn.
    n = model["n"]
    # Chỉ giữ đúng độ dài ngữ cảnh mà mô hình n-gram cần.
    context = tuple(context[-(n - 1):]) if n > 1 else tuple()

    # Add-k smoothing cộng k vào tử số và phân bổ k * |V| vào mẫu số để tránh xác suất 0.
    numerator = model["ngram_counts"][context + (word,)] + k
    denominator = model["context_counts"][context] + k * len(vocabulary)
    if denominator == 0:
        return 0.0
    return numerator / denominator


def sentence_log_probability(model, sentence, vocabulary, k=0.0):
    # TODO: Sum log probabilities for each generated token, including </s>.
    # Dùng tổng log probability thay cho tích xác suất để tránh số quá nhỏ khi câu dài.
    # Nếu không smoothing và gặp xác suất 0, trả về float("-inf").
    n = model["n"]
    padded = ["<s>"] * (n - 1) + tokenize(sentence) + ["</s>"]

    logp = 0.0

    for i in range(n - 1, len(padded)):
        context = tuple(padded[i - n + 1:i])
        word = padded[i]

        p = next_word_probability(
            model,
            context,
            word,
            vocabulary,
            k=k
        )

        if p == 0:
            return float("-inf")

        logp += math.log(p)

    return logp

In [4]:
# Thêm ký hiệu bắt đầu/kết thúc câu để mô hình học được biên câu.
tokenized = [["<s>"] + tokenize(sentence) + ["</s>"] for sentence in corpus]

# Set comprehension gom tất cả token khác nhau; sorted giúp thứ tự từ vựng ổn định.
vocab = sorted({word for sent in tokenized for word in sent})
bigram = train_ngram(corpus, n=2)

# Các assert bên dưới là test tự động: nếu sai, Python sẽ dừng ở dòng assert tương ứng.
assert tokenize("The Wumpus is near.") == ["the", "wumpus", "is", "near", "."]
assert bigram["context_counts"][("the",)] == 11

p_is_after_wumpus = next_word_probability(bigram, ("wumpus",), "is", vocab, k=0.0)
assert abs(p_is_after_wumpus - (2 / 3)) < 1e-9

# Câu này có bigram chưa từng thấy, nên bản không smoothing phải cho log probability là -inf.
raw_logp = sentence_log_probability(bigram, "the agent is glittering .", vocab, k=0.0)
smooth_logp = sentence_log_probability(bigram, "the agent is glittering .", vocab, k=0.5)
assert raw_logp == float("-inf")
assert smooth_logp > float("-inf")

print("Exercise 1 tests passed.")

Exercise 1 tests passed.


## Exercise 2: Bag-of-Words and Naive Bayes

A bag-of-words model ignores word order. Implement a small multinomial Naive Bayes classifier and observe one limitation: `"not good"` still contains the positive clue `"good"`.

In [5]:
train_docs = [
    ("good fun bright", "positive"),
    ("good helpful clear", "positive"),
    ("excellent fun useful", "positive"),
    ("bad boring dull", "negative"),
    ("bad confusing slow", "negative"),
    ("dull boring useless", "negative"),
]


def bow_vector(text):
    # TODO: Return Counter({word: count}) for alphabetic tokens only.
    # Counter giúp biểu diễn bag-of-words: chỉ đếm số lần xuất hiện, không lưu thứ tự từ.
    # Gợi ý: lọc bỏ token không bắt đầu bằng chữ cái bằng re.match(...).
    return Counter(w for w in tokenize(text) if re.match(r"[a-z]", w))


def train_multinomial_nb(examples):
    # TODO: Return enough statistics for prediction:
    # labels, label_counts, word_counts_by_label, total_words_by_label, vocabulary.
    # Gợi ý: cần đếm prior P(label) và likelihood P(word | label) cho từng nhãn.
    # word_counts_by_label nên là dictionary ánh xạ label -> Counter(words).
    labels = sorted({label for _, label in examples})
    label_counts = Counter(label for _, label in examples)
    word_counts = {label: Counter() for label in labels}
    total_words = Counter()
    vocab = set()

    for text, label in examples:
        words = [w for w in tokenize(text) if re.match(r"[a-z]", w)]

        # Lưu số lần mỗi từ xuất hiện theo từng nhãn để ước lượng P(word | label).
        word_counts[label].update(words)
        total_words[label] += len(words)
        vocab.update(words)

    return labels, label_counts, word_counts, total_words, sorted(vocab)


def nb_predict(model, text, alpha=1.0):
    # TODO: Return (best_label, score_dictionary).
    # Gợi ý: tính log prior + tổng log likelihood cho từng nhãn, rồi chọn nhãn có score lớn nhất.
    # alpha là hệ số Laplace smoothing để từ chưa gặp không làm xác suất bằng 0.
    labels, label_counts, word_counts, total_words, vocab = model
    words = [w for w in tokenize(text) if re.match(r"[a-z]", w)]
    scores = {}
    for label in labels:
        # Bắt đầu bằng log prior P(label).
        score = math.log(label_counts[label] / sum(label_counts.values()))
        for word in words:
            # Cộng log likelihood với Laplace smoothing để từ hiếm/không thấy không làm xác suất bằng 0.
            score += math.log((word_counts[label][word] + alpha) / (total_words[label] + alpha * len(vocab)))
        scores[label] = score
    return max(scores, key=scores.get), scores

In [6]:
assert bow_vector("Dog bites dog!") == Counter({"dog": 2, "bites": 1})

nb_model = train_multinomial_nb(train_docs)

# Hai test này kiểm tra mô hình phân loại đúng các câu có từ khóa rõ ràng.
label, scores = nb_predict(nb_model, "good useful fun")
assert label == "positive"

label, scores = nb_predict(nb_model, "bad dull slow")
assert label == "negative"

# Ví dụ này cho thấy hạn chế của bag-of-words: "not good" vẫn chứa từ "good".
print("Prediction for 'not good':", nb_predict(nb_model, "not good")[0])
print("Exercise 2 tests passed.")

Prediction for 'not good': positive
Exercise 2 tests passed.


## Exercise 3: POS Tagging with Viterbi Decoding

Implement Viterbi decoding for a tiny hidden Markov model. The model should choose a likely tag sequence for `"time flies like an arrow"`.

In [9]:
states = ["N", "V", "DET", "P"]

# start_prob: xác suất tag đầu tiên của câu.
start_prob = {"N": 0.35, "V": 0.10, "DET": 0.45, "P": 0.10}

# transition_prob[prev][state]: xác suất chuyển từ tag trước sang tag hiện tại.
transition_prob = {
    "N": {"N": 0.05, "V": 0.55, "DET": 0.05, "P": 0.20, "END": 0.15},
    "V": {"N": 0.20, "V": 0.05, "DET": 0.35, "P": 0.30, "END": 0.10},
    "DET": {"N": 0.90, "V": 0.02, "DET": 0.02, "P": 0.03, "END": 0.03},
    "P": {"N": 0.35, "V": 0.03, "DET": 0.57, "P": 0.02, "END": 0.03},
}

# emission_prob[state][word]: xác suất tag sinh ra word quan sát được.
emission_prob = {
    "N": {"time": 0.30, "flies": 0.20, "arrow": 0.30, "fruit": 0.20},
    "V": {"time": 0.15, "flies": 0.30, "like": 0.30, "saw": 0.25},
    "DET": {"an": 0.50, "the": 0.50},
    "P": {"like": 0.80, "with": 0.20},
}


def safe_log(x):
    # Trả về -inf cho xác suất 0 để phép cộng log vẫn biểu diễn được đường đi không thể xảy ra.
    return math.log(x) if x > 0 else float("-inf")


def viterbi(words):
    # TODO: Return (tagged_words, best_log_score).
    # tagged_words should look like [("time", "N"), ("flies", "V"), ...].
    # Gợi ý: table lưu score tốt nhất đến từng state tại mỗi vị trí.
    # back lưu state trước đó để truy vết lại chuỗi tag tốt nhất sau khi kết thúc.
    table = []
    back = []

    first = {}
    first_back = {}
    for state in states:
        emission = emission_prob[state].get(words[0], 0.001)
        first[state] = safe_log(start_prob[state]) + safe_log(emission)  # ← safe_log
        first_back[state] = None
    table.append(first)
    back.append(first_back)

    for i in range(1, len(words)):
        row = {}
        row_back = {}
        for state in states:
            emission = emission_prob[state].get(words[i], 0.001)
            candidates = []
            for prev in states:
                score = (
                    table[i - 1][prev]
                    + safe_log(transition_prob[prev][state])  # ← safe_log
                    + safe_log(emission)                      # ← safe_log
                )
                candidates.append((score, prev))
            row[state], row_back[state] = max(candidates)
        table.append(row)
        back.append(row_back)

    final_candidates = [
        (table[-1][state] + safe_log(transition_prob[state]["END"]), state)  # ← safe_log
        for state in states
    ]
    best_score, best_state = max(final_candidates)

    tags = [best_state]
    for i in range(len(words) - 1, 0, -1):
        tags.append(back[i][tags[-1]])
    tags.reverse()
    return list(zip(words, tags)), best_score

In [10]:
tagged, score = viterbi(tokenize("time flies like an arrow"))
print(tagged)

# Test này kiểm tra chuỗi tag tốt nhất mà Viterbi phải tìm được từ mô hình xác suất đã cho.
assert tagged == [("time", "N"), ("flies", "V"), ("like", "P"), ("an", "DET"), ("arrow", "N")]
assert isinstance(score, float)
print("Exercise 3 tests passed.")

[('time', 'N'), ('flies', 'V'), ('like', 'P'), ('an', 'DET'), ('arrow', 'N')]
Exercise 3 tests passed.


## Exercise 4: PCFG and Bottom-Up Chart Parsing

Implement a CYK-like chart parser for this small probabilistic context-free grammar. The sentence
`"i saw the wumpus with the telescope"` should have two parses because the prepositional phrase can attach in two different places.

In [11]:
pcfg_rules = [
    ("S",  ("NP", "VP"), 1.00),
    ("NP", ("Det", "N"), 0.45),
    ("NP", ("NP", "PP"), 0.25),
    ("NP", ("i",), 0.30),
    ("VP", ("V", "NP"), 0.60),
    ("VP", ("VP", "PP"), 0.40),
    ("PP", ("P", "NP"), 1.00),
    ("Det", ("the",), 1.00),
    ("N", ("wumpus",), 0.35),
    ("N", ("telescope",), 0.25),
    ("N", ("pit",), 0.20),
    ("N", ("gold",), 0.20),
    ("V", ("saw",), 0.70),
    ("V", ("smelled",), 0.30),
    ("P", ("with",), 0.60),
    ("P", ("near",), 0.40),
]


def split_rules(rules):
    # TODO: Split grammar rules into lexical and binary rules.
    # Return (lexical_rules, binary_rules).
    # Gợi ý: luật lexical có dạng A -> word; luật binary có dạng A -> B C.
    # Nên đổi probability sang math.log(prob) để cộng điểm khi ghép cây parse.
    nonterminals = {lhs for lhs, _, _ in rules}
    lexical = []
    binary = []
    for lhs, rhs, prob in rules:
        if len(rhs) == 1 and rhs[0] not in nonterminals:
            lexical.append((lhs, rhs[0], math.log(prob)))
        elif len(rhs) == 2:
            binary.append((lhs, rhs[0], rhs[1], math.log(prob)))
        else:
            raise ValueError(f"Unsupported rule: {lhs} -> {rhs}")
    return lexical, binary


def tree_to_string(tree, indent=0):
    label, children = tree
    pad = "  " * indent
    if isinstance(children, str):
        return f"{pad}({label} {children})"

    # Đệ quy in cây parse: mỗi node con được thụt vào sâu hơn một mức.
    lines = [f"{pad}({label}"]
    for child in children:
        lines.append(tree_to_string(child, indent + 1))
    lines[-1] += ")"
    return "\n".join(lines)


def chart_parse(words, lexical_rules, binary_rules, keep_per_symbol=5):
    # TODO: Fill a chart over spans and keep the highest-scoring parses.
    # Gợi ý: chart[i][j] nên lưu các parse tốt nhất cho đoạn words[i:j].
    # Với mỗi span, thử mọi điểm tách k và mọi luật binary A -> B C để ghép cây trái/phải.
    # keep_per_symbol giới hạn số cây giữ lại cho mỗi nonterminal để chart không quá lớn.
    n = len(words)

    # chart[i][j] lưu các cây parse tốt nhất cho đoạn words[i:j].
    chart = [[defaultdict(list) for _ in range(n + 1)] for _ in range(n)]

    for i, word in enumerate(words):
        for lhs, terminal, logprob in lexical_rules:
            if terminal == word:
                chart[i][i + 1][lhs].append((logprob, (lhs, word)))

    for span in range(2, n + 1):
        for i in range(n - span + 1):
            j = i + span
            for k in range(i + 1, j):
                # Thử mọi điểm tách k để ghép constituent bên trái và bên phải.
                for lhs, left, right, rule_logprob in binary_rules:
                    for left_logprob, left_tree in chart[i][k].get(left, []):
                        for right_logprob, right_tree in chart[k][j].get(right, []):
                            total = rule_logprob + left_logprob + right_logprob
                            chart[i][j][lhs].append((total, (lhs, [left_tree, right_tree])))

            for symbol in list(chart[i][j].keys()):
                # Giữ lại một số cây tốt nhất cho mỗi nonterminal để bảng không phình quá lớn.
                chart[i][j][symbol].sort(key=lambda item: item[0], reverse=True)
                chart[i][j][symbol] = chart[i][j][symbol][:keep_per_symbol]

    return chart

In [12]:
lexical_rules, binary_rules = split_rules(pcfg_rules)
words = tokenize("i saw the wumpus with the telescope")
chart = chart_parse(words, lexical_rules, binary_rules)
parses = chart[0][len(words)]["S"]

# Câu này có hai cách gắn cụm giới từ "with the telescope", nên cần có 2 parse.
assert len(parses) == 2

# Parse đầu tiên phải có log probability cao hơn parse thứ hai.
assert parses[0][0] > parses[1][0]

for rank, (logprob, tree) in enumerate(parses, 1):
    print("\nParse", rank, "log probability:", round(logprob, 3))
    print(tree_to_string(tree))

print("Exercise 4 tests passed.")


Parse 1 log probability: -7.532
(S
  (NP i)
  (VP
    (VP
      (V saw)
      (NP
        (Det the)
        (N wumpus)))
    (PP
      (P with)
      (NP
        (Det the)
        (N telescope)))))

Parse 2 log probability: -8.002
(S
  (NP i)
  (VP
    (V saw)
    (NP
      (NP
        (Det the)
        (N wumpus))
      (PP
        (P with)
        (NP
          (Det the)
          (N telescope))))))
Exercise 4 tests passed.


## Exercise 5: Augmented Grammar with Agreement Features

A plain CFG may accept sentences such as `"she eat bananas"`. Add agreement features so the checker accepts only compatible subject-verb pairs.

In [13]:
pronoun_features = {
    "i": {"person_number": "1sg"},
    "you": {"person_number": "2"},
    "we": {"person_number": "pl"},
    "they": {"person_number": "pl"},
    "she": {"person_number": "3sg"},
    "he": {"person_number": "3sg"},
}

# Mỗi động từ lưu tập đặc trưng chủ ngữ được phép đi kèm.
verb_features = {
    "eat": {"allowed_subjects": {"1sg", "2", "pl"}},
    "eats": {"allowed_subjects": {"3sg"}},
    "see": {"allowed_subjects": {"1sg", "2", "pl"}},
    "sees": {"allowed_subjects": {"3sg"}},
}

nouns = {"bananas", "gold", "wumpus", "pit"}


def agreement_parse(sentence):
    # TODO: Return (accepted_bool, explanation_string).
    # Gợi ý: tách câu thành subject, verb, object; kiểm tra từng từ có trong lexicon không.
    # Sau đó so sánh person_number của subject với allowed_subjects của verb.
    words = tokenize(sentence)
    if len(words) != 3:
        return False, "Expected: pronoun verb object"

    subj, verb, obj = words
    if subj not in pronoun_features:
        return False, "Unknown subject"
    if verb not in verb_features:
        return False, "Unknown verb"
    if obj not in nouns:
        return False, "Unknown object"

    # So khớp đặc trưng person-number của chủ ngữ với tập chủ ngữ mà động từ cho phép.
    subject_pn = pronoun_features[subj]["person_number"]
    allowed = verb_features[verb]["allowed_subjects"]
    if subject_pn not in allowed:
        return False, f"Agreement failure: subject is {subject_pn}, but verb '{verb}' allows {sorted(allowed)}"

    return True, f"Valid: NP({subject_pn}) + VP({verb})"

In [14]:
# Các assert này kiểm tra cả trường hợp đúng ngữ pháp và sai agreement.
assert agreement_parse("i eat bananas")[0] is True
assert agreement_parse("she eats bananas")[0] is True
assert agreement_parse("she eat bananas")[0] is False
assert agreement_parse("they sees gold")[0] is False

for sent in ["i eat bananas", "she eats bananas", "she eat bananas", "they sees gold"]:
    print(sent, "->", agreement_parse(sent))

print("Exercise 5 tests passed.")

i eat bananas -> (True, 'Valid: NP(1sg) + VP(eat)')
she eats bananas -> (True, 'Valid: NP(3sg) + VP(eats)')
she eat bananas -> (False, "Agreement failure: subject is 3sg, but verb 'eat' allows ['1sg', '2', 'pl']")
they sees gold -> (False, "Agreement failure: subject is pl, but verb 'sees' allows ['3sg']")
Exercise 5 tests passed.


## Exercise 6: Semantic Interpretation

Implement a tiny compositional interpreter for subject-verb-object sentences. The output should be a logical form string such as `Loves(Ali, Bo)`.

In [16]:
names = {"ali": "Ali", "bo": "Bo", "cy": "Cy"}


def love_relation(obj):
    # Hàm trả về lambda để biểu diễn ý tưởng: động từ nhận object rồi chờ subject.
    return lambda subj: f"Loves({subj}, {obj})"


def see_relation(obj):
    return lambda subj: f"Sees({subj}, {obj})"


verb_meanings = {
    "loves": love_relation,
    "sees": see_relation,
}


def interpret_svo(sentence):
    # TODO: Convert "ali loves bo" into "Loves(Ali, Bo)".
    # Gợi ý: tokenize câu thành 3 phần, tra tên trong names, rồi áp dụng verb_meanings[verb](object)(subject).
    words = tokenize(sentence)
    if len(words) != 3:
        raise ValueError("Expected exactly: Name Verb Name")
    subj_word, verb_word, obj_word = words
    subj = names[subj_word]
    obj = names[obj_word]

    # Áp dụng nghĩa của động từ theo kiểu compositional semantics: verb(object)(subject).
    return verb_meanings[verb_word](obj)(subj)

for sentence in ["ali loves bo", "bo loves ali", "cy sees bo"]:
    print(sentence, "=>", interpret_svo(sentence))

ali loves bo => Loves(Ali, Bo)
bo loves ali => Loves(Bo, Ali)
cy sees bo => Sees(Cy, Bo)


In [17]:
assert interpret_svo("ali loves bo") == "Loves(Ali, Bo)"
assert interpret_svo("bo loves ali") == "Loves(Bo, Ali)"
assert interpret_svo("cy sees bo") == "Sees(Cy, Bo)"

facts = {"Loves(Ali, Bo)", "Sees(Cy, Bo)"}

# Sau khi câu được đổi sang logical form, việc kiểm tra đúng/sai là phép membership trong set facts.
print("Is 'ali loves bo' true?", interpret_svo("ali loves bo") in facts)
print("Exercise 6 tests passed.")

Is 'ali loves bo' true? True
Exercise 6 tests passed.


## Exercise 7: Quantifier-Scope Ambiguity

The sentence `"Every robot saw a pit"` has at least two readings:

- narrow-scope existential: every robot saw at least one pit, possibly different pits
- wide-scope existential: there is one pit that every robot saw

Implement both truth conditions.

In [18]:
robots = {"r1", "r2"}
pits = {"p1", "p2"}

world_a = {("r1", "p1"), ("r2", "p2")}
world_b = {("r1", "p1"), ("r2", "p1")}


def every_robot_saw_a_pit_narrow(world):
    # TODO: Return True if each robot saw at least one pit.
    # Gợi ý: dùng all(...) cho "every robot" và any(...) cho "a pit".
    # Ở narrow scope, mỗi robot có thể thấy một pit khác nhau.
    return all(any((robot, pit) in world for pit in pits) for robot in robots)


def every_robot_saw_a_pit_wide(world):
    # TODO: Return True if there exists one pit that every robot saw.
    # Gợi ý: dùng any(...) cho "there exists one pit" bọc ngoài all(...) cho "every robot".
    # Ở wide scope, phải có cùng một pit được tất cả robot nhìn thấy.
    return any(all((robot, pit) in world for robot in robots) for pit in pits)

In [19]:
# world_a đúng theo narrow scope nhưng sai theo wide scope vì hai robot thấy hai pit khác nhau.
assert every_robot_saw_a_pit_narrow(world_a) is True
assert every_robot_saw_a_pit_wide(world_a) is False

# world_b đúng cả hai cách đọc vì cả hai robot đều thấy p1.
assert every_robot_saw_a_pit_narrow(world_b) is True
assert every_robot_saw_a_pit_wide(world_b) is True

print("Exercise 7 tests passed.")

Exercise 7 tests passed.


## Exercise 8: Information Extraction and Question Answering

Extract structured facts from short documents. Then answer questions from those facts.

In [20]:
documents = [
    "Alice grabbed the gold in room1.",
    "Bob smelled the wumpus in room2.",
    "Alice saw a pit in room3.",
    "The breeze means a pit is near room3.",
]

patterns = [
    # Named groups (?P<name>...) giúp lấy trực tiếp agent/action/object/location từ câu.
    ("event", re.compile(r"(?P<agent>[A-Z][a-z]+) (?P<action>grabbed|smelled|saw) (?:the|a) (?P<object>[a-z]+) in (?P<location>room[0-9]+)", re.I)),
    ("clue", re.compile(r"the (?P<clue>breeze|stench) means (?:the|a) (?P<object>[a-z]+) is near (?P<location>room[0-9]+)", re.I)),
]

def extract_facts(docs):
    # TODO: Return a list of dictionaries for event facts and clue facts.
    # Event example:
    # {"kind": "event", "agent": "alice", "action": "grabbed", "object": "gold", "location": "room1"}
    # Clue example:
    # {"kind": "clue", "clue": "breeze", "object": "pit", "location": "room3"}
    # Gợi ý: dùng re.compile(...) với named groups (?P<name>...) để trích các trường cần lưu.
    # groupdict() có thể chuyển named groups thành dictionary để tạo fact nhanh hơn.
    facts = []
    for doc in docs:
        for kind, pattern in patterns:
            match = pattern.search(doc)
            if match:
                # groupdict() chuyển các named groups thành dictionary; ** dùng để trộn vào fact mới.
                item = {"kind": kind, **{k: v.lower() for k, v in match.groupdict().items()}}
                facts.append(item)
                break
    return facts


def answer_question(question, facts):
    # TODO: Support:
    # "What did Alice grab?"
    # "Where did Bob smell the wumpus?"
    # "What is near room3?"
    # Gợi ý: chuẩn hóa question về chữ thường, dùng re.match(...) nhận diện mẫu câu hỏi.
    # Sau đó lọc danh sách facts bằng list comprehension để lấy object/location phù hợp.
    q = question.lower()

    m = re.match(r"what did ([a-z]+) (grab|smell|see)\?", q)
    if m:
        agent, action = m.groups()
        action_map = {"grab": "grabbed", "smell": "smelled", "see": "saw"}
        wanted = action_map[action]

        # List comprehension lọc các fact khớp agent và action, rồi lấy object làm câu trả lời.
        answers = [f["object"] for f in facts if f.get("agent") == agent and f.get("action") == wanted]
        return ", ".join(answers) if answers else "I do not know."

    m = re.match(r"where did ([a-z]+) (grab|smell|see) (?:the|a) ([a-z]+)\?", q)
    if m:
        agent, action, obj = m.groups()
        action_map = {"grab": "grabbed", "smell": "smelled", "see": "saw"}
        wanted = action_map[action]
        answers = [
            f["location"]
            for f in facts
            if f.get("agent") == agent and f.get("action") == wanted and f.get("object") == obj
        ]
        return ", ".join(answers) if answers else "I do not know."

    m = re.match(r"what is near (room[0-9]+)\?", q)
    if m:
        location = m.group(1)
        answers = [f["object"] for f in facts if f["kind"] == "clue" and f["location"] == location]
        return ", ".join(answers) if answers else "I do not know."

    return "I do not understand the question pattern."

In [21]:
facts = extract_facts(documents)
assert len(facts) == 4

# Các test này kiểm tra cả câu hỏi có câu trả lời và câu hỏi không có fact phù hợp.
assert answer_question("What did Alice grab?", facts) == "gold"
assert answer_question("Where did Bob smell the wumpus?", facts) == "room2"
assert answer_question("What is near room3?", facts) == "pit"
assert answer_question("What did Bob grab?", facts) == "I do not know."

for fact in facts:
    print(fact)

print("Exercise 8 tests passed.")

{'kind': 'event', 'agent': 'alice', 'action': 'grabbed', 'object': 'gold', 'location': 'room1'}
{'kind': 'event', 'agent': 'bob', 'action': 'smelled', 'object': 'wumpus', 'location': 'room2'}
{'kind': 'event', 'agent': 'alice', 'action': 'saw', 'object': 'pit', 'location': 'room3'}
{'kind': 'clue', 'clue': 'breeze', 'object': 'pit', 'location': 'room3'}
Exercise 8 tests passed.


## Reflection Questions

Answer these in a markdown cell:

1. Why does smoothing matter for n-gram language models?
2. What kind of information is lost in a bag-of-words representation?
3. Why can a sentence have more than one parse tree?
4. What do augmented grammars add beyond plain CFG categories?
5. Why do real NLP systems usually combine statistical, syntactic, semantic, and pragmatic evidence?

**Answer:**
1. Without smoothing, any unseen n-gram gets probability 0, making the entire sentence probability 0 — smoothing redistributes mass to unseen events so the model generalizes beyond training data.

2. Word order and syntactic structure are lost — "dog bites man" and "man bites dog" become identical representations.

3. Natural language is inherently ambiguous — the same word sequence can satisfy multiple grammatical rules simultaneously (e.g., "I saw the man with the telescope" has two valid attachments for the PP).

4. Augmented grammars attach features (agreement, case, subcategorization) to categories, allowing finer-grained constraints that plain CFG rules cannot express without exploding the rule set.

5. No single layer captures all meaning — statistical evidence handles frequency and likelihood, syntax provides structure, semantics provides meaning, and pragmatics resolves context-dependent ambiguity; each fills gaps the others cannot.